In [1]:
"""
Примеры кода к Лекции 6.1: От одного агента к системе агентов

Этот модуль демонстрирует:
1. Ограничения одиночного агента (один промпт на все роли)
2. Базовая мультиагентная система + сравнение с одиночным агентом
3. Анатомия мультиагентного состояния (эволюция state через агентов)
"""

'\nПримеры кода к Лекции 6.1: От одного агента к системе агентов\n\nЭтот модуль демонстрирует:\n1. Ограничения одиночного агента (один промпт на все роли)\n2. Базовая мультиагентная система + сравнение с одиночным агентом\n3. Анатомия мультиагентного состояния (эволюция state через агентов)\n'

In [2]:
import operator
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph

from llm_config import check_api_key, get_llm

In [3]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


In [4]:
# ============================================================================
# ВСПОМОГАТЕЛЬНЫЕ ИНСТРУМЕНТЫ
# ============================================================================
@tool
def web_search(query: str) -> str:
    """Поиск информации в интернете по запросу."""
    # Имитация поискового API — в реальности здесь был бы Tavily, Serper и т.д.
    knowledge = {
        "квантовые компьютеры": (
            "Квантовые компьютеры используют кубиты вместо классических битов. "
            "Google Willow (2024) достиг 105 кубитов. IBM планирует 100 000 кубитов к 2033. "
            "Основные области применения: криптография, моделирование молекул, оптимизация. "
            "Квантовое превосходство впервые продемонстрировано Google в 2019 году."
        ),
        "искусственный интеллект": (
            "ИИ переживает бум благодаря трансформерам и LLM. "
            "GPT-4 и Claude демонстрируют способности к рассуждению. "
            "AI-агенты — следующий этап: автономные системы, решающие задачи. "
            "Рынок ИИ оценивается в $200 млрд к 2025 году."
        ),
    }
    # Ищем по ключевым словам
    for key, value in knowledge.items():
        if key in query.lower():
            return value
    return f"Результаты поиска по запросу '{query}': информация не найдена в демо-базе."

In [5]:
# ============================================================================
# ЧАСТЬ 1: ОГРАНИЧЕНИЯ ОДИНОЧНОГО АГЕНТА
# ============================================================================


def example_single_agent_limits():
    """
    Один агент пытается быть исследователем, писателем и редактором одновременно.

    Демонстрирует проблему размытого фокуса: когда в одном промпте
    смешаны инструкции для разных ролей, агент справляется с каждой
    посредственно. Промпт перегружен, и модель вынуждена балансировать
    между противоречивыми требованиями.
    """
    print("=" * 60)
    print("ПРИМЕР 1: Ограничения одиночного агента")
    print("=" * 60)

    llm = get_llm()

    # Перегруженный промпт: три роли в одном
    overloaded_prompt = """Ты выполняешь три роли одновременно:

РОЛЬ 1 — ИССЛЕДОВАТЕЛЬ:
- Найди ключевые факты по теме
- Укажи конкретные цифры, даты, имена
- Проверь достоверность каждого факта

РОЛЬ 2 — ПИСАТЕЛЬ:
- Напиши связную статью на основе найденных фактов
- Используй живой, увлекательный стиль
- Структурируй текст с подзаголовками

РОЛЬ 3 — РЕДАКТОР:
- Проверь текст на фактические ошибки
- Исправь стилистические шероховатости
- Убедись, что каждое утверждение подкреплено фактом

Выполни все три роли последовательно и верни финальный результат."""

    # Состояние для одиночного агента
    class SingleAgentState(TypedDict):
        task: str
        result: str

    def solo_agent(state: SingleAgentState) -> dict:
        """Узел: один агент делает всё."""
        response = llm.invoke(
            [
                SystemMessage(content=overloaded_prompt),
                HumanMessage(content=f"Тема: {state['task']}"),
            ]
        )
        return {"result": response.content}

    # Граф из одного узла
    graph = StateGraph(SingleAgentState)
    graph.add_node("solo", solo_agent)
    graph.add_edge(START, "solo")
    graph.add_edge("solo", END)
    app = graph.compile()

    result = app.invoke({"task": "Квантовые компьютеры в 2024 году"})

    print("\n--- Результат одиночного агента ---")
    print(f"Длина ответа: {len(result['result'])} символов")
    print(f"Превью: {result['result'][:300]}...")
    print("\nПроблемы:")
    print("  - Один промпт на 3 роли → размытый фокус")
    print("  - Агент не может использовать инструменты для исследования")
    print("  - Нет проверки качества другим «глазом»")
    print("  - Все токены контекста тратятся на один мега-промпт")

    return result["result"]


if __name__ == "__main__":
    example_single_agent_limits()

ПРИМЕР 1: Ограничения одиночного агента

--- Результат одиночного агента ---
Длина ответа: 7602 символов
Превью: # Квантовые компьютеры в 2024 году: от лабораторных рекордов к практическим задачам

В 2024 году квантовые компьютеры остаются одной из самых обсуждаемых технологий в мире — но уже не как абстрактное обещание будущего, а как поле реальных инженерных прорывов, жесткой конкуренции и осторожного, но за...

Проблемы:
  - Один промпт на 3 роли → размытый фокус
  - Агент не может использовать инструменты для исследования
  - Нет проверки качества другим «глазом»
  - Все токены контекста тратятся на один мега-промпт


In [6]:
# ============================================================================
# ЧАСТЬ 2: БАЗОВАЯ МУЛЬТИАГЕНТНАЯ СИСТЕМА
# ============================================================================


class TeamState(TypedDict):
    """Состояние мультиагентной команды.

    Каждый агент читает нужные поля и обновляет своё:
    - researcher пишет в research
    - writer пишет в draft
    - editor пишет в feedback и final_text
    """

    task: str
    research: str
    draft: str
    feedback: str
    final_text: str


def example_basic_multi_agent():
    """
    Три специализированных агента в линейном графе + сравнение с одиночным агентом.

    TeamState с полями: task, research, draft, feedback, final_text.
    Три узла: researcher_node → writer_node → editor_node.
    Каждый агент — сфокусированный промпт и одна задача.

    Архитектура: START → researcher → writer → editor → END
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 2: Базовая мультиагентная система")
    print("=" * 60)

    llm = get_llm()

    # --- A) Одиночный агент для сравнения ---
    print("\n--- A) Одиночный агент ---")
    solo_response = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Ты — универсальный ассистент. Исследуй тему, напиши статью "
                    "и отредактируй её. Верни финальный результат."
                )
            ),
            HumanMessage(
                content="Напиши краткую статью о квантовых компьютерах в 2024 году."
            ),
        ]
    )
    solo_result = solo_response.content
    print(f"  Длина: {len(solo_result)} символов")
    print(f"  Превью: {solo_result[:200]}...")

    # --- B) Мультиагентная команда ---
    print("\n--- B) Мультиагентная команда ---")

    # --- Агент 1: Исследователь ---
    def researcher_node(state: TeamState) -> dict:
        """Исследователь: ищет факты по теме."""
        # В реальной системе здесь был бы create_agent с инструментом поиска.
        # Для простоты используем прямой вызов инструмента + LLM.
        raw_facts = web_search.invoke({"query": state["task"]})

        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "Ты — исследователь. Твоя единственная задача — "
                        "систематизировать найденные факты. Выдели ключевые тезисы, "
                        "укажи цифры и даты. Не пиши статью — только факты."
                    )
                ),
                HumanMessage(
                    content=f"Тема: {state['task']}\n\nНайденные данные:\n{raw_facts}"
                ),
            ]
        )
        print(f"\n  [Исследователь] Найдены факты ({len(response.content)} символов)")
        return {"research": response.content}

    # --- Агент 2: Писатель ---
    def writer_node(state: TeamState) -> dict:
        """Писатель: создаёт черновик статьи на основе фактов."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "Ты — талантливый автор статей. Напиши увлекательную, "
                        "структурированную статью на основе предоставленных фактов. "
                        "Используй подзаголовки. Не выдумывай факты — работай "
                        "только с тем, что тебе дали."
                    )
                ),
                HumanMessage(
                    content=(
                        f"Тема: {state['task']}\n\n"
                        f"Факты от исследователя:\n{state['research']}"
                    )
                ),
            ]
        )
        print(f"  [Писатель] Черновик готов ({len(response.content)} символов)")
        return {"draft": response.content}

    # --- Агент 3: Редактор ---
    def editor_node(state: TeamState) -> dict:
        """Редактор: проверяет и улучшает черновик."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "Ты — строгий редактор. Проверь статью на:\n"
                        "1. Фактические ошибки (сверься с исходными фактами)\n"
                        "2. Стилистические проблемы\n"
                        "3. Логику изложения\n\n"
                        "Выдай исправленную финальную версию статьи. "
                        "Если статья хорошая — верни её с минимальными правками."
                    )
                ),
                HumanMessage(
                    content=(
                        f"Исходные факты:\n{state['research']}\n\n"
                        f"Черновик статьи:\n{state['draft']}"
                    )
                ),
            ]
        )
        print(f"  [Редактор] Финальный текст готов ({len(response.content)} символов)")
        return {
            "final_text": response.content,
            "feedback": "Статья проверена и отредактирована.",
        }

    # --- Сборка графа ---
    graph = StateGraph(TeamState)

    graph.add_node("researcher", researcher_node)
    graph.add_node("writer", writer_node)
    graph.add_node("editor", editor_node)

    graph.add_edge(START, "researcher")
    graph.add_edge("researcher", "writer")
    graph.add_edge("writer", "editor")
    graph.add_edge("editor", END)

    app = graph.compile()

    # Запуск
    result = app.invoke(
        {
            "task": "Квантовые компьютеры в 2024 году",
            "research": "",
            "draft": "",
            "feedback": "",
            "final_text": "",
        }
    )

    print(f"\n--- Финальный результат команды ---")
    print(f"Длина: {len(result['final_text'])} символов")
    print(f"Превью: {result['final_text'][:300]}...")

    # --- Сравнение ---
    print("\n--- Сравнение ---")
    print(f"  Одиночный агент: {len(solo_result)} символов")
    print(f"  Команда:         {len(result['final_text'])} символов")
    print("  Команда обычно даёт более структурированный текст:")
    print("    - Факты проверены отдельным этапом (исследователь)")
    print("    - Стиль вычитан отдельным этапом (редактор)")
    print("    - Цена: ~3x больше токенов на межагентную коммуникацию")

    return result["final_text"]


if __name__ == "__main__":
    example_basic_multi_agent()


ПРИМЕР 2: Базовая мультиагентная система

--- A) Одиночный агент ---
  Длина: 1513 символов
  Превью: # Квантовые компьютеры в 2024 году: краткий обзор

В 2024 году квантовые компьютеры остаются одной из самых перспективных, но всё ещё экспериментальных технологий. Их главное отличие от обычных компью...

--- B) Мультиагентная команда ---

  [Исследователь] Найдены факты (386 символов)
  [Писатель] Черновик готов (2726 символов)
  [Редактор] Финальный текст готов (2726 символов)

--- Финальный результат команды ---
Длина: 2726 символов
Превью: # Квантовые компьютеры в 2024 году

Квантовые компьютеры остаются одной из самых обсуждаемых технологий современности. В 2024 году интерес к ним особенно высок: отрасль продолжает развиваться, а компании делают новые шаги в наращивании вычислительных возможностей. При этом квантовые системы всё ещё ...

--- Сравнение ---
  Одиночный агент: 1513 символов
  Команда:         2726 символов
  Команда обычно даёт более структурированный текст:
    - Ф

In [7]:
# ============================================================================
# ЧАСТЬ 3: АНАТОМИЯ МУЛЬТИАГЕНТНОГО СОСТОЯНИЯ
# ============================================================================


def example_shared_state():
    """
    Эволюция разделяемого состояния при прохождении через агентов.

    Показывает TeamState с разными типами полей:
    - task: str (неизменяемый вход)
    - messages: Annotated[list, operator.add] (аккумулируемый лог)
    - agent_outputs: dict (каждый агент пишет свой результат)
    - current_agent: str (поле маршрутизации)

    Демонстрирует, как состояние эволюционирует на каждом шаге:
    после каждого агента показывает, какие поля изменились.
    """
    print("\n" + "=" * 60)
    print("ПРИМЕР 3: Анатомия мультиагентного состояния")
    print("=" * 60)

    # Состояние с разными типами полей
    class DetailedTeamState(TypedDict):
        task: str  # Неизменяемый вход
        messages: Annotated[list[str], operator.add]  # Аккумулируемый лог
        agent_outputs: dict  # Словарь результатов
        current_agent: str  # Текущий агент
        iteration: int  # Счётчик шагов

    llm = get_llm()

    def analyst_node(state: DetailedTeamState) -> dict:
        """Аналитик: анализирует задачу и формирует план."""
        response = llm.invoke(
            [
                SystemMessage(
                    content="Ты — аналитик. Кратко проанализируй задачу и предложи план из 3 пунктов."
                ),
                HumanMessage(content=state["task"]),
            ]
        )
        plan = response.content

        # Обновляем несколько полей одновременно
        outputs = dict(state.get("agent_outputs", {}))
        outputs["analyst"] = plan

        return {
            "messages": [f"[Аналитик] План составлен: {len(plan)} символов"],
            "agent_outputs": outputs,
            "current_agent": "analyst",
            "iteration": state.get("iteration", 0) + 1,
        }

    def executor_node(state: DetailedTeamState) -> dict:
        """Исполнитель: реализует план аналитика."""
        plan = state["agent_outputs"].get("analyst", "Нет плана")

        response = llm.invoke(
            [
                SystemMessage(
                    content="Ты — исполнитель. Кратко выполни план и опиши результат."
                ),
                HumanMessage(content=f"Задача: {state['task']}\nПлан:\n{plan}"),
            ]
        )
        result = response.content

        outputs = dict(state.get("agent_outputs", {}))
        outputs["executor"] = result

        return {
            "messages": [f"[Исполнитель] Результат готов: {len(result)} символов"],
            "agent_outputs": outputs,
            "current_agent": "executor",
            "iteration": state.get("iteration", 0) + 1,
        }

    def reviewer_node(state: DetailedTeamState) -> dict:
        """Рецензент: оценивает результат исполнителя."""
        executor_result = state["agent_outputs"].get("executor", "Нет результата")

        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "Ты — рецензент. Оцени результат по шкале 1-10 "
                        "и дай краткий комментарий (1-2 предложения)."
                    )
                ),
                HumanMessage(
                    content=f"Задача: {state['task']}\nРезультат:\n{executor_result}"
                ),
            ]
        )
        review = response.content

        outputs = dict(state.get("agent_outputs", {}))
        outputs["reviewer"] = review

        return {
            "messages": [f"[Рецензент] Оценка готова: {review[:80]}"],
            "agent_outputs": outputs,
            "current_agent": "reviewer",
            "iteration": state.get("iteration", 0) + 1,
        }

    # --- Сборка графа ---
    graph = StateGraph(DetailedTeamState)

    graph.add_node("analyst", analyst_node)
    graph.add_node("executor", executor_node)
    graph.add_node("reviewer", reviewer_node)

    graph.add_edge(START, "analyst")
    graph.add_edge("analyst", "executor")
    graph.add_edge("executor", "reviewer")
    graph.add_edge("reviewer", END)

    app = graph.compile()

    # --- Запуск с отслеживанием эволюции состояния ---
    print("\nЗапуск с stream_mode='updates' для отслеживания изменений:\n")

    initial_state = {
        "task": "Составить обзор трендов AI в 2024 году",
        "messages": ["[Система] Задача поставлена"],
        "agent_outputs": {},
        "current_agent": "",
        "iteration": 0,
    }

    # Используем streaming для отслеживания изменений на каждом шаге
    for chunk in app.stream(initial_state, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"--- После узла: {node_name} ---")
            if "current_agent" in update:
                print(f"  current_agent: {update['current_agent']}")
            if "iteration" in update:
                print(f"  iteration: {update['iteration']}")
            if "messages" in update:
                for msg in update["messages"]:
                    print(f"  messages += {msg}")
            if "agent_outputs" in update:
                keys = list(update["agent_outputs"].keys())
                print(f"  agent_outputs: ключи = {keys}")
            print()

    # Финальное состояние
    final = app.invoke(initial_state)
    print("--- Финальное состояние ---")
    print(f"  task: {final['task']}")
    print(f"  current_agent: {final['current_agent']}")
    print(f"  iteration: {final['iteration']}")
    print(f"  messages ({len(final['messages'])} шт.):")
    for msg in final["messages"]:
        print(f"    - {msg}")
    print(f"  agent_outputs: {list(final['agent_outputs'].keys())}")


if __name__ == "__main__":
    example_shared_state()


ПРИМЕР 3: Анатомия мультиагентного состояния

Запуск с stream_mode='updates' для отслеживания изменений:

--- После узла: analyst ---
  current_agent: analyst
  iteration: 1
  messages += [Аналитик] План составлен: 644 символов
  agent_outputs: ключи = ['analyst']

--- После узла: executor ---
  current_agent: executor
  iteration: 2
  messages += [Исполнитель] Результат готов: 4929 символов
  agent_outputs: ключи = ['analyst', 'executor']

--- После узла: reviewer ---
  current_agent: reviewer
  iteration: 3
  messages += [Рецензент] Оценка готова: 8/10

Обзор хорошо структурирован, охватывает ключевые направления 2024 года и д
  agent_outputs: ключи = ['analyst', 'executor', 'reviewer']

--- Финальное состояние ---
  task: Составить обзор трендов AI в 2024 году
  current_agent: reviewer
  iteration: 3
  messages (4 шт.):
    - [Система] Задача поставлена
    - [Аналитик] План составлен: 940 символов
    - [Исполнитель] Результат готов: 2987 символов
    - [Рецензент] Оценка готова: 